In [7]:
import pandas as pd 
import numpy as np 

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/clean_supply_chain.csv")

df.head()


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,Customer State,Customer Zipcode,Department Name,Latitude,Longitude,Market,Order City,Order Country,order date (DateOrders),Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,725.0,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,1/31/2018 22:56,13.110000,0.04,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,59405.0,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,725.0,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,1/13/2018 12:27,16.389999,0.05,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,59405.0,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,Consumer,CA,95125.0,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,1/13/2018 12:06,18.030001,0.06,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,59405.0,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,Home Office,CA,90027.0,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,1/13/2018 11:45,22.940001,0.07,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,59405.0,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,Corporate,PR,725.0,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,1/13/2018 11:24,29.500000,0.09,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,59405.0,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [8]:
df["Late_delivery_risk"].value_counts()

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

In [9]:
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])

In [10]:
df["order month"] = (
    df["order date (DateOrders)"].dt.month
)

In [11]:
df["order_dayOfWeek"] = (
    df["order date (DateOrders)"].dt.weekday
    
)

In [13]:
df["is_weekend"] = (
    df["order_dayOfWeek"] >=5 
).astype(int)

In [14]:
df["order_total_value"] =  (
    df["Order Item Product Price"] * df["Order Item Quantity"]
)

In [15]:
df["Is_International"] = (
    df["Customer Country"] != df["Order Country"]
)

In [17]:
new_features = [
    "order month",
    "order_dayOfWeek",
    "is_weekend",
    "order_total_value",
    "Is_International"
]

df[new_features].head(10)

,order month,order_dayOfWeek,is_weekend,order_total_value,Is_International
0,1,2,0,327.75,True
1,1,5,1,327.75,True
2,1,5,1,327.75,True
3,1,5,1,327.75,True
4,1,5,1,327.75,True
5,1,5,1,327.75,True
6,1,5,1,327.75,True
7,1,5,1,327.75,True
8,1,5,1,327.75,True
9,1,5,1,327.75,True


In [18]:
df[new_features].isnull().sum()

order month          0
order_dayOfWeek      0
is_weekend           0
order_total_value    0
Is_International     0
dtype: int64

In [19]:
leakage_columns = [
    "Days for shipping (real)",
    "Delivery Status",
    "shipping date (DateOrders)",
    "Order Status"
]

df.drop(columns=leakage_columns , inplace=True)

In [20]:
df.drop( columns=["order date (DateOrders)"],
    inplace=True)

In [25]:
# طباعة كل 5 أسماء في سطر واحد
cols = df.columns.tolist()
for i in range(0, len(cols), 5):
    chunk = cols[i:i+5]
    print(f"{i+1}-{i+len(chunk)}: " + " | ".join(chunk))

1-5: Type | Days for shipment (scheduled) | Benefit per order | Sales per customer | Late_delivery_risk
6-10: Category Name | Customer City | Customer Country | Customer Segment | Customer State
11-15: Customer Zipcode | Department Name | Latitude | Longitude | Market
16-20: Order City | Order Country | Order Item Discount | Order Item Discount Rate | Order Item Product Price
21-25: Order Item Profit Ratio | Order Item Quantity | Sales | Order Item Total | Order Profit Per Order
26-30: Order Region | Order State | Order Zipcode | Product Name | Product Price
31-35: Product Status | Shipping Mode | order month | order_dayOfWeek | is_weekend
36-37: order_total_value | Is_International


In [26]:
X = df.drop(columns=["Late_delivery_risk"])

Y = df["Late_delivery_risk"]

In [27]:
X.shape

(180519, 36)

In [28]:
Y.shape

(180519,)

In [29]:
"Late_delivery_risk" in X.columns

False

In [30]:
df.to_csv(
    "../data/processed/featured_supply_chain.csv",
    index=False
)